# Load Pre-trained Embedding Model

In [7]:
from sentence_transformers import SentenceTransformer
from datasets import load_dataset, concatenate_datasets
import torch

model_id = "Snowflake/snowflake-arctic-embed-m"  # Use a reasonably good model here

model_retrieval = SentenceTransformer(
    model_id, device="cuda" if torch.cuda.is_available() else "cpu"
)

# Load and Combine Multiple Datasets

In [8]:
from datasets import load_dataset, concatenate_datasets

# Load each split separately
easy_para = load_dataset("frankwong2001/ssf-dataset_Full_synthetic_v2", "easy_triplets_paraphrase")["train"]
hard_para = load_dataset("frankwong2001/ssf-dataset_Full_synthetic_v2", "hard_triplets_paraphrase")["train"]
# hard_sem = load_dataset("frankwong2001/ssf-dataset_Full_synthetic_batch10", "hard_triplets_semantic")["train"]

# Concatenate all splits into one dataset
dataset = concatenate_datasets([easy_para, hard_para,])

# Clean Dataset Structure

In [9]:
dataset = dataset.select_columns(['anchor', 'positive', 'negative'])
dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 3770
})

# Define Quality Assessment Functions
### These functions compute semantic similarity scores between different text pairs in our triplets:
- get_embeddings(): Converts text to vector representations using our embedding model
- get_similarities(): Computes cosine similarity between embedding vectors
- format_data_retriever(): Processes batches of triplets to add similarity scores

### The similarity scores help us identify high-quality triplets where:
- Anchor-positive pairs have high similarity (good matches)
- Anchor-negative pairs have moderate similarity (hard negatives)
- Positive-negative pairs are sufficiently different

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

def get_embeddings(texts):
    vectors = model_retrieval.encode(texts)
    return [vector.tolist() for vector in vectors]


def get_similarities(vector_batch_a, vector_batch_b):
    similarities = []
    for vector_a, vector_b in zip(vector_batch_a, vector_batch_b):
        similarity = cosine_similarity([vector_a], [vector_b])[0][0]
        similarities.append(similarity)
    return similarities

def format_data_retriever(batch):# -&gt; Any:
    batch["anchor-vector"] = get_embeddings(batch["anchor"])
    batch["positive-vector"] = get_embeddings(batch["positive"])
    batch["negative-vector"] = get_embeddings(batch["negative"])    
    batch["similarity-positive-negative"] = get_similarities(batch["positive-vector"], batch["negative-vector"])
    batch["similarity-anchor-positive"] = get_similarities(batch["anchor-vector"], batch["positive-vector"])
    batch["similarity-anchor-negative"] = get_similarities(batch["anchor-vector"], batch["negative-vector"])
    return batch

# Compute Similarity Scores

In [11]:
dataset = dataset.map(format_data_retriever, batched=True, batch_size=250)

Map:   0%|          | 0/3770 [00:00<?, ? examples/s]

# Inspect the Enriched Dataset

In [12]:
dataset.to_pandas()

,anchor,positive,negative,anchor-vector,positive-vector,negative-vector,similarity-positive-negative,similarity-anchor-positive,similarity-anchor-negative
0,The Audit Associate/Audit Assistant Associate ...,The Audit Associate plays a crucial role in ex...,The Tax Associate is responsible for managing ...,"[0.06092929095029831, 0.09926585108041763, 0.0...","[0.033896420150995255, 0.08134564757347107, 0....","[0.055267173796892166, 0.06705760210752487, 0....",0.770916,0.911135,0.719227
1,The Audit Senior Manager/Audit Manager manages...,The Audit Senior Manager oversees a diverse ra...,The Tax Associate manages a variety of complia...,"[0.057781320065259933, 0.07172407954931259, -0...","[0.035467613488435745, 0.0701984241604805, -0....","[0.04586879163980484, 0.06939025968313217, 0.0...",0.599612,0.948275,0.552682
2,The Audit Partner/Audit Director is a transfor...,The Audit Director is a visionary leader who g...,The Tax Associate is responsible for ensuring ...,"[0.03437991812825203, 0.07796177268028259, -0....","[0.033775508403778076, 0.05857576057314873, -0...","[0.051393091678619385, 0.08724277466535568, 0....",0.599007,0.859856,0.531721
3,The Audit Senior is expected to team lead vari...,The Audit Senior is responsible for leading di...,The Tax Associate is tasked with managing vari...,"[0.02032621204853058, 0.06756345182657242, -0....","[0.006788385100662708, 0.07294884324073792, -0...","[0.05090216174721718, 0.0878765732049942, 0.01...",0.646537,0.913284,0.652562
4,The Business Valuation Associate/Business Valu...,The Business Valuation Executive plays a cruci...,The Business Valuation Associate is tasked wit...,"[0.032666150480508804, 0.05040699988603592, -0...","[0.025334952399134636, 0.04067045822739601, -0...","[0.0420728363096714, 0.06408200412988663, -0.0...",0.789803,0.911506,0.835593
...,...,...,...,...,...,...,...,...,...
3765,The WSH Manager is responsible for reviewing W...,The WSH Manager oversees the evaluation of wor...,The WSH Manager is responsible for developing ...,"[-0.013950522057712078, 0.08462847769260406, -...","[-0.010776442475616932, 0.07027486711740494, -...","[-0.029965145513415337, 0.09673049300909042, -...",0.776481,0.940672,0.823297
3766,The WSH Officer is responsible for developing ...,The WSH Officer is tasked with creating and ov...,The WSH Officer is in charge of managing the o...,"[-0.0010848849778994918, 0.05022452771663666, ...","[-0.010148255154490471, 0.0433991365134716, -0...","[0.016622763127088547, 0.06001840531826019, -0...",0.878950,0.939121,0.836560
3767,The Workplace Safety and Health (WSH) Supervis...,Description\nThe Workplace Safety and Health (...,Description\nThe Workplace Safety and Health (...,"[0.020672127604484558, 0.08999120444059372, -0...","[0.030078833922743797, 0.09906085580587387, -0...","[0.04915937781333923, 0.06600330024957657, -0....",0.834527,0.942158,0.847850
3768,The Lead Workplace Safety and Health (WSH) Aud...,The Lead Workplace Safety and Health (WSH) Aud...,The Lead Workplace Safety and Health (WSH) Coo...,"[-0.003643264528363943, 0.04361746087670326, -...","[-0.003616597270593047, 0.05639877915382385, -...","[0.030149031430482864, 0.06779344379901886, -0...",0.894869,0.945745,0.848915


# Apply Quality Filtering Criteria
### Filtering Rules:
1. Anchor-Positive similarity > 0.6: Ensures positive examples are semantically related to anchors
2. Anchor-Negative similarity 0.2-0.6: Creates "hard negatives" that are somewhat related but not too similar
3. Positive-Negative similarity < 0.7: Ensures positive and negative examples are sufficiently distinct

In [ ]:
def filter_with_hard_negatives(example):
    anchor_pos = example["similarity-anchor-positive"]
    anchor_neg = example["similarity-anchor-negative"] 
    pos_neg = example["similarity-positive-negative"]
    
    # return (
    #     anchor_pos > 0.6 and  # Good positive
    #     0.2 < anchor_neg < 0.6 and  # Hard negative range
    #     pos_neg < 0.7  # Positive and negative are distinct
    # )

    return (
        anchor_pos > 0.78 and  # Good positive
        0.2 < anchor_neg < 0.78 and  # Hard negative range
        pos_neg < 0.7  # Positive and negative are distinct (ask dickson why need this)
    )


cleaned_dataset = dataset.filter(filter_with_hard_negatives)

Filter:   0%|          | 0/5655 [00:00<?, ? examples/s]

# Clean Filtered Dataset
### Remove the embedding vectors and similarity scores to keep only the essential text data. This reduces storage requirements while preserving the high-quality triplets identified by our filtering process.

In [9]:
cleaned_dataset = cleaned_dataset.select_columns(['anchor', 'positive', 'negative'])
cleaned_dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 1407
})

# Split into Training and Validation Sets

In [10]:
train_size = int(0.8 * len(cleaned_dataset))
valid_size = len(cleaned_dataset) - train_size

train_dataset = cleaned_dataset.select(range(train_size))
valid_dataset = cleaned_dataset.select(range(train_size, train_size + valid_size))

# Verify Dataset Splits

In [11]:
train_dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 1125
})

In [12]:
valid_dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 282
})

In [13]:
from datasets import DatasetDict

ds = DatasetDict({
    "train": train_dataset,
    "valid": valid_dataset
})

ds

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 1125
    })
    valid: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 282
    })
})

# Create Dataset Dictionary

In [14]:
ds.push_to_hub("frankwong2001/ssf-train-valid-full-synthetic-batch10-cleaned")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  45%|####5     |  527kB / 1.16MB            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        : 100%|##########|  240kB /  240kB            

CommitInfo(commit_url='https://huggingface.co/datasets/frankwong2001/ssf-train-valid-full-synthetic-batch10-cleaned/commit/7ed5d68d8ca15ee4c388940a248a69b5d8f345b9', commit_message='Upload dataset', commit_description='', oid='7ed5d68d8ca15ee4c388940a248a69b5d8f345b9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/frankwong2001/ssf-train-valid-full-synthetic-batch10-cleaned', endpoint='https://huggingface.co', repo_type='dataset', repo_id='frankwong2001/ssf-train-valid-full-synthetic-batch10-cleaned'), pr_revision=None, pr_num=None)